# Collider-bias correction with SlopeHunter

A progression GWAS is run in cases only, so conditioning on having the disease
opens a path between any risk variant and the progression outcome: a variant
that raises risk looks associated with progression even when it is not. This
is collider bias, and it is corrected here by estimating the
slope of that induced relationship and subtracting it.

Three estimators are run on the same data to compare methods:

* **SlopeHunter** (Mahmoud et al. 2022, *Nat Commun*) - clusters variants and
  fits the slope on the ones that act only on risk;
* **Dudbridge et al. (2019)** - weighted regression with a SIMEX correction;
* **Cai et al. (2022)** `ColliderBias` - CWLS and related estimators.

**Inputs:** a disease-risk (incidence) GWAS and a progression (cases-only)
GWAS, both in LocusZoom format from `locuszoom_prep.ipynb`.
**Output:** progression statistics with adjusted beta, SE and P.

**Kernel: R** (IRkernel).

## 0. Packages - run once

```r
remotes::install_github("Osmahmoud/SlopeHunter")   # SlopeHunter
remotes::install_github("DudbridgeLab/indexevent") # Dudbridge et al. 2019
remotes::install_github("qingyuanzhao/mr.raps")    # dependency of ColliderBias
remotes::install_github("SiyangCai/ColliderBias")  # Cai et al. 2022
install.packages("R.utils")                        # for reading .gz
```

In [ ]:
suppressPackageStartupMessages({
  library(SlopeHunter)
  library(data.table)
})

## 1. Inputs and parameters

Paths come from the environment so nothing machine-specific is stored here.

In [ ]:
inc_file  <- Sys.getenv("SH_INCIDENCE_FILE")    # disease-risk GWAS
prog_file <- Sys.getenv("SH_PROGNOSIS_FILE")    # progression GWAS, cases only
out_pref  <- Sys.getenv("SH_OUT_PREFIX", unset = "progression_SH_adjusted")

# 1000 Genomes PLINK prefix for local LD clumping. Downloaded once with:
#   curl -L -o 1kg.v3.tgz http://fileserve.mrcieu.ac.uk/ld/1kg.v3.tgz
#   tar -xzvf 1kg.v3.tgz        # gives EUR, AFR, AMR, EAS, SAS .bed/.bim/.fam
ref_bfile <- Sys.getenv("SH_REF_BFILE")

for (v in c("inc_file", "prog_file")) {
  if (!nzchar(get(v))) stop("set SH_INCIDENCE_FILE and SH_PROGNOSIS_FILE")
}

xp_thresh <- 0.001   # incidence P threshold for variants used to fit the slope
M_boot    <- 100     # bootstrap replicates; use ~1000 for a final run
seed      <- 2021
gw        <- 5e-8    # genome-wide significance, for the before/after table

## 2. Read the summary statistics

Adjust the `*_col` arguments if the input headers differ.

In [ ]:
incidence <- read_incidence(
  inc_file,
  snp_col           = "rsid",
  beta_col          = "beta",
  se_col            = "stderr_beta",
  eaf_col           = "alt_allele_freq",
  effect_allele_col = "alt",
  other_allele_col  = "ref",
  chr_col           = "#chrom",
  pos_col           = "pos"
)

prognosis <- read_prognosis(
  prog_file,
  snp_col           = "rsid",
  beta_col          = "beta",
  se_col            = "stderr_beta",
  eaf_col           = "alt_allele_freq",
  effect_allele_col = "alt",
  other_allele_col  = "ref",
  chr_col           = "#chrom",
  pos_col           = "pos"
)

cat("incidence:", nrow(incidence), "variants | prognosis:", nrow(prognosis), "\n")

In [ ]:
# Drop implausible progression estimates: a huge beta or SE means the model
# did not converge for that variant, and rare variants are unstable.
# Tighten to 0.03/0.97 for a more conservative run.
prognosis <- subset(prognosis,
  abs(BETA.prognosis) < 3 &
  SE.prognosis < 1 &
  EAF.prognosis >= 0.01 & EAF.prognosis <= 0.99)

cat("prognosis after QC:", nrow(prognosis), "variants\n")
summary(prognosis$BETA.prognosis)
summary(prognosis$SE.prognosis)

## 3. Merge and harmonise

Aligns the progression effects to the incidence effect allele. `remove` flags
variants that could not be harmonised.

In [ ]:
Sp.Data <- harmonise_effects(incidence, prognosis)
cat("variants after merge/harmonisation:", nrow(Sp.Data), "\n")
names(Sp.Data)

In [ ]:
table(Sp.Data$remove, useNA = "ifany")

Sp.Data <- Sp.Data[!Sp.Data$remove %in% TRUE, ]
cat("variants after dropping flagged ones:", nrow(Sp.Data), "\n")

## 4. Thin the fitting set

Run LD prune on input variant set.

### Local reference panel

In [ ]:
Sp.Fit <- as.data.frame(LD_prune(
  as.data.frame(Sp.Data),
  clump_kb  = 50, clump_r2 = 0.1,
  Random    = TRUE, clump_p1 = 1,
  local     = TRUE, ref_bfile = ref_bfile,
  seed      = 77777
))

cat("variants after pruning:", nrow(Sp.Fit), "\n")
cat("passing xp_thresh:", sum(Sp.Fit$PVAL.incidence < xp_thresh, na.rm = TRUE), "\n")
cat("fraction in the reference panel:",
    mean(Sp.Data$SNP %in% fread(paste0(ref_bfile, ".bim"))$V2), "\n")

## 5. Estimate the slope

`hunt()` separates variants that act only on risk from those that also act on
progression, and fits the correction slope on the first group.

In [ ]:
Sp.Model <- hunt(
  dat              = as.data.frame(Sp.Fit),
  xp_thresh        = xp_thresh,
  init_pi          = 0.6,
  init_sigmaIP     = 1e-5,
  Bootstrapping    = TRUE,
  M                = M_boot,
  seed             = seed,
  Plot             = TRUE,
  show_adjustments = FALSE
)

print(Sp.Model)

b   <- Sp.Model$b
bse <- Sp.Model$bse
cat(sprintf("SlopeHunter slope b = %.4f (SE %.4f)\n", b, bse))

## 6. Apply the correction to every variant

In [ ]:
adj <- SHadj(
  Sp.Model,
  as.data.frame(Sp.Data),
  snp_col   = "SNP",
  xbeta_col = "BETA.incidence", xse_col = "SE.incidence",
  ybeta_col = "BETA.prognosis", yse_col = "SE.prognosis"
)
head(adj)

In [ ]:
# Before and after for one variant.
check_snp <- "rs11599166"
row <- adj[adj$SNP == check_snp, ]

cat("SNP:", check_snp, "\n")
cat(sprintf("  risk P           %.3g\n", row$PVAL.incidence))
cat(sprintf("  progression P    %.3g  ->  adjusted %.3g\n", row$PVAL.prognosis, row$yp.adj))
cat(sprintf("  progression beta %.4f  ->  adjusted %.4f\n", row$ybeta, row$ybeta.adj))
cat(sprintf("  progression SE   %.4f  ->  adjusted %.4f\n", row$yse, row$yse.adj))

In [ ]:
# Unadjusted against adjusted: points should sit near the diagonal, with the
# spread growing for variants that have a strong risk effect.
options(repr.plot.width = 6, repr.plot.height = 6)
plot(adj$ybeta, adj$ybeta.adj,
     pch = 16, cex = 0.4, col = adjustcolor("grey30", 0.4),
     xlab = "Unadjusted progression beta", ylab = "Adjusted progression beta")
abline(0, 1, lty = 2, col = "red")

In [ ]:
fwrite(adj, paste0(out_pref, ".tsv.gz"), sep = "\t")
fwrite(data.table(b = b, bse = bse, pi = Sp.Model$pi, entropy = Sp.Model$entropy),
       paste0(out_pref, "_slope.tsv"), sep = "\t")
cat("wrote", paste0(out_pref, ".tsv.gz"), "and", paste0(out_pref, "_slope.tsv"), "\n")

## 7. Other estimators

Same data, different slope estimators. Agreement between them is the main
evidence that the correction is trustworthy.

### Weighted linear regression (most simple, naive version)

In [ ]:
f <- Sp.Fit[Sp.Fit$PVAL.incidence < xp_thresh, ]
summary(lm(BETA.prognosis ~ BETA.incidence, data = f,
           weights = 1 / SE.prognosis^2))$coefficients

### Dudbridge et al. (2019)

https://github.com/DudbridgeLab/indexevent

In [ ]:
library(indexevent)

cols <- c("BETA.incidence", "SE.incidence", "BETA.prognosis", "SE.prognosis")
f2 <- f[complete.cases(f[, cols]) & apply(f[, cols], 1, function(r) all(is.finite(r))), ]
cat(nrow(f2), "variants in the fitting set\n")

ie <- indexevent(f2$BETA.incidence, f2$SE.incidence,
                 f2$BETA.prognosis, f2$SE.prognosis,
                 weighted = TRUE, method = "Hedges-Olkin")
cat(sprintf("Dudbridge slope b = %.4f (SE %.4f), CI [%.4f, %.4f]\n",
            ie$b, ie$b.se, ie$b.ci[1], ie$b.ci[2]))

In [ ]:
# Apply it to every variant, using the pruned set as the fitting subset.
d <- as.data.frame(Sp.Data)
d <- d[complete.cases(d[, cols]) & apply(d[, cols], 1, function(r) all(is.finite(r))), ]
idx <- which(d$SNP %in% f2$SNP)

ie_all <- indexevent(d$BETA.incidence, d$SE.incidence,
                     d$BETA.prognosis, d$SE.prognosis,
                     weighted = TRUE, prune = idx, method = "Hedges-Olkin")

d$BETA.prognosis.adj <- ie_all$ybeta.adj
d$SE.prognosis.adj   <- ie_all$yse.adj
d$PVAL.prognosis.adj <- ie_all$yp.adj

want <- c("SNP", "CHR.prognosis", "POS.prognosis", "BETA.incidence",
          "BETA.prognosis", "SE.prognosis", "PVAL.prognosis",
          "BETA.prognosis.adj", "SE.prognosis.adj", "PVAL.prognosis.adj")
head(d[order(d$PVAL.prognosis.adj), want], 20)

### Cai et al. (2022)

https://github.com/SiyangCai/ColliderBias

In [ ]:
library(ColliderBias)

cb <- methodCB(d$BETA.incidence, d$SE.incidence,
               d$BETA.prognosis, d$SE.prognosis,
               prune = idx, method = "CWLS", weighted = "1", model = "ivw")
cat(sprintf("CWLS slope b = %.4f (SE %.4f), CI [%.4f, %.4f]\n",
            cb$b, cb$b.se, cb$b.ci[1], cb$b.ci[2]))

In [ ]:
# Compare the available estimators side by side.
run_cb <- function(method, model = "ivw") {
  tryCatch({
    r <- methodCB(d$BETA.incidence, d$SE.incidence, d$BETA.prognosis, d$SE.prognosis,
                  prune = idx, method = method, model = model)
    data.frame(method = method, model = model, b = r$b, se = r$b.se,
               lo = r$b.ci[1], hi = r$b.ci[2], b.raw = r$b.raw)
  }, error = function(e) {
    message(method, " (", model, ") failed: ", conditionMessage(e))
    data.frame(method = method, model = model, b = NA, se = NA,
               lo = NA, hi = NA, b.raw = NA)
  })
}

comp <- rbind(run_cb("CWLS"), run_cb("CWLS", "egger"), run_cb("SH"))

# MR-RAPS directly, without the shrinkage step
raps <- mr.raps::mr.raps.mle(d$BETA.incidence[idx], d$BETA.prognosis[idx],
                             d$SE.incidence[idx],   d$SE.prognosis[idx],
                             over.dispersion = FALSE)
comp <- rbind(comp, data.frame(method = "mr.raps.mle", model = "ivw",
                               b = raps$beta.hat, se = raps$beta.se,
                               lo = raps$beta.hat - 1.96 * raps$beta.se,
                               hi = raps$beta.hat + 1.96 * raps$beta.se,
                               b.raw = NA))
comp

In [ ]:
# What the correction changes at genome-wide significance.
d$BETA.prognosis.adj <- cb$ybeta.adj
d$SE.prognosis.adj   <- cb$yse.adj
d$PVAL.prognosis.adj <- cb$yp.adj

table(unadjusted = d$PVAL.prognosis < gw, adjusted = d$PVAL.prognosis.adj < gw)

adj_sig <- d[!is.na(d$PVAL.prognosis.adj) & d$PVAL.prognosis.adj < gw, ]
adj_sig[order(adj_sig$PVAL.prognosis.adj), want]

In [ ]:
fwrite(d, paste0(out_pref, "_ColliderBias_CWLS.tsv.gz"), sep = "\t")
fwrite(comp, paste0(out_pref, "_slope_comparison.tsv"), sep = "\t")
cat("wrote the ColliderBias outputs\n")